# RAG(Retrieval-Augmented Generation)
- <font color=red>기존의 LLM을 확장하여 주어진 컨텍스트나 질문에 대해 더욱 정확하고 풍부한 정보를 제공</font>하는 방법
- 즉, 모델이 학습 데이터에 포함되지 않은 외부 데이터를 실시간으로 검색(retrieval)하고 검색된 데이터를 활용(augmented)하여 이를 바탕으로 답변을 생성(generation)하는 것

- 기본 구조
  - <font color=red>검색 단계(Retrieval Phase)</font>: 사용자의 질문이나 컨텍스트를 입력으로 받아서, 이와 관련된 외부 데이터를 검색하는 단계
  - <font color=red>증강 단계(Augmented Phase)</font>: 검색된 데이터를 토큰화, 인코딩, 임베딩 후에 벡터 DB에 저장하고 검색기를 붙이는 단계  
  - <font color=red>생성 단계(Generation Phase)</font>: 벡터 DB에 저장된 데이터와 LLM 모델을 사용하여 사용자의 질문에 답변을 생성하는 단계

- 장점  
  - <font color=red>풍부한 정보 제공</font> : RAG 모델은 검색을 통해 얻은 외부 데이터를 활용하여, 보다 구체적이고 풍부한 정보를 제공
  - <font color=red>실시간 정보 반영</font> : 최신 데이터를 검색하여 반영함으로써, 모델이 실시간으로 변화하는 정보에 대응
  - <font color=red>환각 방지</font> : 검색을 통해 실제 데이터에 기반한 답변을 생성함으로써, 환각 현상이 발생할 위험을 줄이고 정확도 향상

- RAG 8단계 프로세스
  - 사전 준비 단계
    - 문서 가져오기
    - 텍스트 분할
    - 임베딩
    - 벡터 DB에 저장
  - Runtime 단계
    - 검색기 설정
    - 프롬프트 구성
    - LLM 모델 객체 생성
    - 체인 생성 및 실행

In [ ]:
# open AI API key 등록
import os
OPENAI_API_KEY=""
# 현재 노트북 커널에 환경변수 등록
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY


# LangSmith API key 등록
LANGSMITH_TRACING="true" # 추적 할지 말지
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_API_KEY=""
LANGSMITH_PROJECT="Langchain0422"
# 현재 노트북 커널에 환경변수 등록
os.environ['LANGSMITH_TRACING'] = LANGSMITH_TRACING
os.environ['LANGSMITH_ENDPOINT'] = LANGSMITH_ENDPOINT
os.environ['LANGSMITH_API_KEY'] = LANGSMITH_API_KEY
os.environ['LANGSMITH_PROJECT'] = LANGSMITH_PROJECT

In [2]:
!pip install langchain langchain-openai langchain-community faiss-cpu pypdf chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180

# RAG 1 : PDF를 학습한 나만의 챗봇 만들기
  - (1) 데이터 로드(Load Data)
    - RAG에 사용할 데이터를 불러오는 단계
  - (2) 텍스트 분할(Text Split)
    - 불러온 데이터를 작은 크기의 단위(chunk)로 분할하는 단계
  - (3) 임베딩 (Embedding) / 인덱싱 (Indexing)
    - 텍스트 데이터를 숫자로 이루어진 벡터로 변환하는 단계
    - 분할된 텍스트를 검색 가능한 형태로 만드는 단계
  - (4) 검색(Retrieval)
    - 사용자의 질문이나 주어진 컨텍스트에 가장 관련된 정보를 찾아내는 단계
  - (5) 생성(Generation)
    - 검색된 정보를 바탕으로 사용자의 질문에 답변을 생성하는 최종 단계

### 0. 기본 LLM 답변 확인하기

In [5]:
from langchain.chat_models import init_chat_model # 모델을 유연하게 생성하는 도구
from langchain_core.output_parsers import StrOutputParser # 문자열 파싱을 해주는 도구

In [6]:
# 모델생성
llm_4o_mini = init_chat_model("openai:gpt-4o-mini", max_tokens=1024)
# 체인구성하기
chain = llm_4o_mini | StrOutputParser()

In [7]:
chain.invoke("KDT 사업제안을 하려고하는데 사업유형은 어떤게 있어?")

'KDT(Korea Digital Transformation) 사업 제안을 위한 사업 유형은 여러 가지가 있습니다. 아래는 몇 가지 주요 사업 유형을 소개합니다.\n\n1. **소프트웨어 개발 및 서비스**:\n   - 디지털 전환을 위한 맞춤형 소프트웨어 개발\n   - 클라우드 서비스 및 인프라 제공\n   - SaaS(Software as a Service) 플랫폼 구축\n\n2. **데이터 분석 및 활용**:\n   - 빅데이터 분석 솔루션\n   - AI 및 머신러닝 기반의 예측 모델 개발\n   - 데이터 시각화 및 경영 대시보드 구축\n\n3. **IoT(Internet of Things) 솔루션**:\n   - IoT 센서 및 장비를 통한 실시간 모니터링 시스템 구축\n   - 스마트 팩토리, 스마트 시티 관련 솔루션\n\n4. **디지털 마케팅**:\n   - 온라인 마케팅 전략 및 실행\n   - SEO(검색 엔진 최적화) 및 콘텐츠 마케팅\n   - 소셜 미디어 관리 및 광고 캠페인\n\n5. **모바일 앱 및 웹 플랫폼**:\n   - 사용자 경험(UX)과 사용자 인터페이스(UI) 개선을 위한 애플리케이션 개발\n   - 전자상거래 플랫폼 설계 및 운영\n\n6. **Cybersecurity**:\n   - 사이버 보안 솔루션 및 컨설팅\n   - 데이터 보호 및 관리 시스템 구축\n\n7. **교육 및 컨설팅**:\n   - 디지털 전환 관련 교육 프로그램 제공\n   - 기업의 디지털 전환 전략 수립 및 컨설팅 서비스\n\n8. **스마트 물류 및 SCM(Supply Chain Management)**:\n   - 물류 자동화 및 최적화 솔루션\n   - 통합 공급망 관리 시스템 구축\n\n위의 유종을 바탕으로 사업 제안을 구체화할 수 있으며, 각 산업 분야별 특성에 맞춘 디지털 전환 솔루션을 제시하는 것도 좋습니다. 무엇보다 타겟 시장과 고객의 요구에 맞춰 제안을 준비하는 것이 중요합니다.'

In [8]:
chain.invoke("KDT 직종별 단가는 얼마야?")

'KDT(한국디지털투자)*의 직종별 단가는 특정한 자료나 보고서에 따라 달라질 수 있으며, 구체적인 숫자는 공개된 자료를 통해 확인해야 정확합니다. 각 직종의 평균 단가, 경력 수준, 지역, 시장 상황 등에 따라 차이가 발생할 수 있습니다. 일반적으로 구직 사이트나 산업별 협회에서 제공하는 자료를 참고하는 것이 좋습니다.\n\n*참고: KDT라는 용어가 특정 조직이나 프로그램을 지칭하는 것일 수 있으므로, 좀 더 구체적인 정보를 제공해 주시면 더 나은 답변을 드릴 수 있습니다.'

### (1) 데이터 로드(Load Data)
  - RAG에 사용할 데이터를 불러오는 단계

In [10]:
# PDF 파일을 읽어오는 라이브러리
from langchain_community.document_loaders import PyPDFLoader

In [12]:
# 파일로더 생성
loader = PyPDFLoader("/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM 활용/data/KDT.pdf")
# 데이터로딩
documents = loader.load()

In [13]:
len(documents)

2

In [15]:
documents[1]

Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2023-12-22T09:41:58+00:00', 'source': '/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM 활용/data/KDT.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='- 4 -\n[ 일반 훈련과정 개요 ] 1. (신기술 과정) 첨단산업·디지털 분야 21개 직종 과정(전체 훈련시간의 30%이상 프로젝트학습으로 편성하여야 함)   ※ 신청 가능 아카데미: 전체 유형 2. (융합분야 과정) 신기술 분야의 세부 직종 간 융합 훈련과정, 디지털 분야와 인문사회 분야의 세부직종을 병행 학습하여 특정분야에 취업하도록 하는 훈련과정 또는 신기술 분야와 전통산업 직종간 융합과정(전체 훈련시간의 30%이상 프로젝트학습으로 편성하여야 함)  - 단, 디지털 분야 직종과 인문 사회 분야 직무 능력이 융합된 훈련 과정은 해당 직무능력이 필요로 하는 취업 분야 제시 필수   ※ 신청 가능 아카데미: 전체 유형 3. (기타분야 과정) 산업현장의 인력 수요가 크고 높은 기술력이 요구되나, 21개 신기술 직종에는 포함되지 않는 경우 예외적으로 허용하는 과정(전체 훈련시간의 30%이상 프로젝트학습으로 편성하여야 함)   ※ 신청 가능 아카데미: 첨단산업 디지털 선도기업 아카데미, 대학 주도형 아카데미 ※ ‘융합분야 과정’, ‘기타분야 과정’ 신청 시 과정명에 융합_과정명, 기타_과정명으로 작성○훈련비지원기준은아래3가지중에서선택하여지원신청 ①(NCS단가*의130%)신기술(21개직종)훈련과정인경우    * 신기술 대분야(디지털, 소재·부품, 로봇·드론 등 6개 분야)와 연계된 NCS 직종의 평균단가를 적용\x00   * 분야별 평균단가: 디지털 7,303원, 소재·부품 7,600원,

### (2) 텍스트 분할(Text Split)
  - 불러온 데이터를 작은 크기의 단위(chunk)로 분할하는 단계

- 텍스트 분할(Text Split)
  - <font color=red>CharacterTextSplitter</font> : 주어진 텍스트를 설정한 단위(문단(\n\n), 문장(\n), 단어(빈공백, 형태소) 등)로 분할
  - <font color=red>RecursiveCharacterTextSplitter</font> : 텍스트를 재귀적으로 분할하여 의미적으로 관련 있는 텍스트 조각들이 같이 있도록 하는 목적으로 설계
    - <font color=red>문자 리스트(['\n\n', '\n', ' ', ''])의 문자를 순서대로 사용하여 텍스트를 분할</font>하며, 분할된 청크들이 설정된 chunk_size보다 작아질 때까지 이 과정을 반복

  - <font color=red>split_documents()</font> : 데이터 로더의 반환값에서 page_content 항목을 찾아서 청크를 분리하는 함수
  - <font color=red>split_text()</font> : 데이터 로더의 반환값에서 page_content 항목만을 가져와서서 청크를 분리하는 함수
  - 반환값 구성
    - page_content : 분할된 토큰이 저장
    - metadata : 원본 문서에 대한 정보가 저장

In [18]:
from langchain_text_splitters import CharacterTextSplitter # 단순 구분자로 자르는 도구
from langchain_text_splitters import RecursiveCharacterTextSplitter # 재귀적으로 여러 구분자로 자르는 도구

In [21]:
# 단순 chart splitter 생성
char_splitter = CharacterTextSplitter(
    separator = " ", # 구분자 설정
    chunk_size = 10, # 기대하는 청크의 최대 사이즈
    chunk_overlap = 5 # 청크끼리 겹치는 구간 크기
)

In [29]:
char_splitter.split_text("안녕하세요333333333333333 오늘 점심은 무엇을 먹으면 좋을까요? 날씨가 화창하니 좋습니다.")

['안녕하세요333333333333333',
 '오늘 점심은 무엇을',
 '무엇을 먹으면',
 '먹으면 좋을까요?',
 '좋을까요? 날씨가',
 '날씨가 화창하니',
 '화창하니 좋습니다.']

In [24]:
# Recursive splitter 사용하기
recursive_splitter = RecursiveCharacterTextSplitter(
    separators = [" ", ""], # 구분자 설정
    chunk_size = 10, # 기대하는 청크의 최대 사이즈
    chunk_overlap = 5 # 청크끼리 겹치는 구간 크기
)

In [28]:
recursive_splitter.split_text("안녕하세요22222222222222222 오늘 점심은 무엇을 먹으면 좋을까요? 날씨가 화창하니 좋습니다.")

['안녕하세요22222',
 '2222222222',
 '2222222222',
 '2222222',
 '오늘 점심은',
 '점심은 무엇을',
 '무엇을 먹으면',
 '먹으면 좋을까요?',
 '날씨가 화창하니',
 '좋습니다.']

#### 최종청크 만들기

In [30]:
kdt_splitter = RecursiveCharacterTextSplitter(
    separators = ['\n\m', '\n', ' ', ''], # separators 생략시 문단, 줄바꿈, 띄어쓰기, 글자 순으로 자르도록 기본 설정 됨
    chunk_size = 500, # 청크의 글자 크기
    chunk_overlap = 100 # 청크 오버랩 크기
)

<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_48467/2025508168.py:2: SyntaxWarning: invalid escape sequence '\m'
  separators = ['\n\m', '\n', ' ', ''], # separators 생략시 문단, 줄바꿈, 띄어쓰기, 글자 순으로 자르도록 기본 설정 됨


In [31]:
kdt_chunk = kdt_splitter.split_documents(documents)

In [32]:
# 청크사이즈 확인
for chunk in kdt_chunk :
  print(len(chunk.page_content))

132
469
5
497
457
379


In [33]:
kdt_chunk[0]

Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2023-12-22T09:41:58+00:00', 'source': '/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM 활용/data/KDT.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='- 3 -\n□ K-디지털 트레이닝 아카데미 유형\uf000디지털신기술아카데미:참여기업의훈련수요를기반으로참여기업과훈련기관간협약을체결한후설계된첨단산업·디지털분야훈련과정\uf000벤처\n스타트업아카데미:기업이필요로하는인재를양성하기위해벤처또는스타트업등의기업이속한협')

### (3) 임베딩 (Embedding) / 인덱싱 (Indexing)
  - 텍스트 데이터를 숫자로 이루어진 벡터로 변환하는 단계
  - 분할된 텍스트를 검색 가능한 형태로 만드는 단계

In [34]:
from langchain_openai import OpenAIEmbeddings # 임베딩 도구
from langchain_community.vectorstores import Chroma # 백터 DB 도구

In [36]:
vec_db = Chroma.from_documents(
    documents = kdt_chunk, # 임베딩할 벡터
    embedding = OpenAIEmbeddings() # 임베딩 도구 연결
)

In [37]:
vec_db

### (4) 검색(Retrieval)
  - 사용자의 질문이나 주어진 컨텍스트에 가장 관련된 정보를 찾아내는 단계

In [40]:
# 사용자 질문과 유사한 청크 2개를 찾는 검색기 생성
retriver = vec_db.as_retriever(search_kwargs={'k': 2})

In [41]:
# retriver 실행 -> runnable 요소
retriver.invoke("KDT 단가는 얼마야?")

[Document(metadata={'producer': 'iLovePDF', 'page': 1, 'moddate': '2023-12-22T09:41:58+00:00', 'total_pages': 2, 'creator': 'PyPDF', 'source': '/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM 활용/data/KDT.pdf', 'creationdate': '', 'page_label': '2'}, page_content='에너지 7,686원 ②(18,150원*)첨단산업분야(12개직종)훈련과정및고성과맞춤형훈련과정인경우신청가능    * 훈련내용 및 훈련 수준 등을 검토하여 NCS단가 130%로 조정 가능-다만,첨단산업디지털선도기업은직종에관계없이단일단가(18,150원)신청가능 ③(NCS단가의300%이내또는훈련비실비*)3D프린팅,로봇,드론직종을제외한첨단산업분야(9개직종)훈련과정인경우신청가능    * NCS 단가의 300% 또는 훈련비 실비 과정은 기존 직업훈련시장에 없는 훈련과정이고 NCS 단가 130%, 18,150원 수준으로 운영이 어려운 과정에 한하여 예외적으로 인정함. 이 경우에도 심사과정에서 훈련내용 및 훈련 수준 등을 검토하여 18,150원 또는 NCS 단가 130%로 조정할 수 있음'),
 Document(metadata={'creator': 'PyPDF', 'moddate': '2023-12-22T09:41:58+00:00', 'page_label': '2', 'total_pages': 2, 'source': '/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM 활용/data/KDT.pdf', 'producer': 'iLovePDF', 'page': 1, 'creationdate': ''}, page_content='포함되지 않는 경우 예외적으로 허용하는 과정(전체 훈련시간의 30%이상 프로젝트학습으로 편성하여야 함)   ※ 신청 가능 아카데미:

### (5) 생성(Generation)
  - 검색된 정보를 바탕으로 사용자의 질문에 답변을 생성하는 최종 단계

In [43]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [44]:
template = ChatPromptTemplate.from_messages([
    ('system', '당신은 직업훈련 관련 질문을 답변하는 챗봇입니다.'),
    ('system', '아래 Context를 참고해서 답변을 해주세요. 모르는 정보는 모른다고 솔직하게 이야기하세요.'),
    ('system', 'context : {context}'),
    ('human', 'question : {question}')
])

In [51]:
# 체인 구성하기
chatbot_chain = {'question' : RunnablePassthrough(),
                 'context' : retriver} | template | llm_4o_mini | StrOutputParser()

In [54]:
print(chatbot_chain.invoke("KDT 사업제안을 하려고하는데 사업유형은 어떤게 있어?"))

KDT 사업 제안 시 선택할 수 있는 사업유형은 다음과 같습니다:

1. **디지털신기술 아카데미**: 참여기업의 훈련 수요를 기반으로 참여기업과 훈련기관 간 협약을 체결한 후 설계된 첨단 산업·디지털 분야 훈련 과정입니다.

2. **벤처 스타트업 아카데미**: 기업이 필요로 하는 인재를 양성하기 위해 벤처 또는 스타트업 기업이 속한 협단체가 회원사의 인력 수요를 조사하고 훈련 기관과 협약을 체결한 후 설계된 첨단 산업·디지털 분야 훈련 과정입니다.

3. **지역·산업 주도형 아카데미**: 지역별 인적 자원 개발 위원회(RSC)나 산업별 인적 자원 개발 위원회(ISC)와 함께 참여기업과 훈련 기관을 발굴·매칭하여 3자 협약을 체결하여 설계, 운영하는 훈련 과정입니다.

4. **첨단 산업 디지털 선도기업 아카데미**: 첨단 산업 디지털 선도 기업이 자체적으로 또는 운영 지원 기관과 함께 운영하는 훈련 과정입니다.

5. **대학 주도형 아카데미**: 고등교육법에 따라 대학 또는 산학협력 촉진에 관한 법률에 의한 산학협력단이 참여기업과 협약을 체결한 후 설계된 첨단 산업·디지털 분야 훈련 과정입니다.

이러한 유형을 고려하여 사업 제안을 준비하시면 됩니다.
